## Imports

In [ ]:
import duckdb
import pandas as pd
from google.cloud import bigquery
from google.colab import auth
from google.colab import syntax
from google.colab import userdata

pd.options.display.float_format = '{:_.2f}'.format

### Provide your credentials

In [ ]:
auth.authenticate_user()
print('Authenticated')

## List projects



In order to query BigQuery, you will need to specify a project ID.  To get a list of project IDs associated with your account, run the following command.

In [ ]:
!gcloud projects list --sort-by=projectId

## Declare the Cloud billing project ID

In [ ]:
billing_project_id = userdata.get('bq_billing_project_id')


In [ ]:
# Create client object
client = bigquery.Client(project=billing_project_id)


## Get Dataset metadata

### How many data sets are in the bigquery-public-data project?


In [ ]:
datasets = client.list_datasets(project="bigquery-public-data")  # Make an API request.
dataset_list = [dataset.dataset_id for dataset in datasets]


In [ ]:
len(dataset_list)


In [ ]:
dataset_list[:10]


### For each data set collect metadata.

In [ ]:
datasets = client.list_datasets(project="bigquery-public-data")  # Make an API request.

dataset_metadata = []  # List to hold the data for the DataFrame

for i, dataset in enumerate(datasets):
    # if i >= 3: break

    fdi = f"{dataset.project}.{dataset.dataset_id}"
    dataset_info = client.get_dataset(fdi)
    print(f"{i:03}: {dataset_info.dataset_id}")

    # Append relevant metadata to the list
    dataset_metadata.append({
        "Dataset ID": dataset_info.dataset_id,
        "Description": dataset_info.description,
        "Location": dataset_info.location,
        "Created Time": dataset_info.created,
        "Last Modified Time": dataset_info.modified,
        "Labels": dataset_info.labels,
        "Access Entries": dataset_info.access_entries,
    })

# Create a DataFrame from the list of data
dataset_metadata_df = pd.DataFrame(dataset_metadata)
dataset_metadata_df[:10]

In [ ]:
dataset_metadata_df.drop(columns=["Access Entries"])[:3]


In [ ]:
parquet_file = "bigquery.dataset.metadata.description.parquet"
dataset_metadata_df.drop(columns=["Access Entries"]).to_parquet( parquet_file, index=False)


In [ ]:
!ls -l big* --si


In [ ]:
duckdb.query(f"select count(1) from '{parquet_file}' ;")


In [ ]:
duckdb.query(f'''
  select *
  from '{parquet_file}'
  limit 10
  ;
''').df()


In [ ]:
stop


## Push parquet to AWS S3


In [ ]:
import os


In [ ]:
%%capture install_output
%%bash
pip install -U awscli


In [ ]:
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('aws_key')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('aws_secret')
os.environ['AWS_DEFAULT_REGION'] = 'us-west-1'


In [ ]:
f"s3://ddc-datascience/{parquet_file}"

In [ ]:
! aws s3 cp {parquet_file} s3://ddc-datascience/{parquet_file}


In [ ]:
url = f"https://ddc-datascience.s3.amazonaws.com/{parquet_file}"
url


In [ ]:
duckdb.query(f'''
  select count(1) as 'number of data sets'
  from '{url}'
  ;
''').df()


In [ ]:
duckdb.query(f'''
  select *
  from '{url}'
  limit 1
;
''').df()


## Load the entire remote parquet file into a data frame

In [ ]:
df = duckdb.query(f"select * from '{url}';").df()
df.shape


In [ ]:
df["Last Modified Time"].max(), df["Last Modified Time"].min()



In [ ]:
df.info()


In [ ]:
import textwrap

for i, row in df.iterrows():
  print( f"=== {i:03}" )
  print( row["Dataset ID"] )
  description = row.get( "Description", "" )
  description = description if description else ""
  for line in description.splitlines():
    print( textwrap.fill( line, width = 150 ) )
  print()


## Experimenting with API requests


experimenting with difference methods.


In [ ]:
## trying to understand
datasets = client.list_datasets(project="bigquery-public-data")  # Make an API request.
for i, dataset in enumerate(datasets):
    if i >= 1: break
[ x for x in dir(dataset) if not x.startswith("_") ]

full_dataset = client.get_dataset(dataset.reference)
df = pd.DataFrame([ ( x, type(getattr(full_dataset, x)).__name__ ) for x in dir(full_dataset) if not x.startswith("_") ]).sort_values(by=[1,0])

import yaml
print(yaml.dump(full_dataset.to_api_repr(), sort_keys=False, default_flow_style=False))


In [ ]:
## trying to understand
datasets = client.list_datasets(project="bigquery-public-data")  # Make an API request.

# dataset_metadata = []  # List to hold the data for the DataFrame

for i, dataset in enumerate(datasets):
    if i >= 3: break

    fdi = f"{dataset.project}.{dataset.dataset_id}"
    dataset_info = client.get_dataset(fdi)

    print(f"{i:03}: {dataset_info.dataset_id}")
    print(f"{i:03}: {dataset.dataset_id}")
    foo = dir(dataset)
    print(len(foo))

#     # Append relevant metadata to the list
#     dataset_metadata.append({
#         "Dataset ID": dataset_info.dataset_id,
#         "Description": dataset_info.description,
#         "Location": dataset_info.location,
#         "Created Time": dataset_info.created,
#         "Last Modified Time": dataset_info.modified,
#         "Labels": dataset_info.labels,
#         "Access Entries": dataset_info.access_entries,
#     })

# # Create a DataFrame from the list of data
# dataset_metadata_df = pd.DataFrame(dataset_metadata)
# dataset_metadata_df[:10]



### For each data set, get table information.

In [ ]:
# dataset_id
tables = client.list_tables(dataset)
try:
  for table in tables:
    print("hi")
except:
  print("nope")


## Get number of rows, columns, and bytes

In [ ]:
datasets = client.list_datasets(project="bigquery-public-data")  # Make an API request.

data = []  # List to hold the data for the DataFrame

for i, dataset in enumerate(datasets):
  # if i >= 3: break
  # if i >= 50: break
  # if i >= 50: break

  dataset_id = dataset.dataset_id
  print(f"{i:03}: {dataset_id}")
  tables = client.list_tables(dataset)  # Make an API request.
  try:
    for table in tables:
      table_ref = client.get_table(table)  # Make an API request.
      table_id = table_ref.table_id
      num_rows = table_ref.num_rows
      num_columns = len(table_ref.schema)
      size_mb = round(table_ref.num_bytes / (1000 ** 2),2)
      modified = table_ref.modified

      # Append the data to the list
      data.append({
        "Dataset ID": dataset_id,
        'Table': table_id,
        'Number of Rows': num_rows,
        'Number of Columns': num_columns,
        'Size in MB': size_mb,
        'Last Modified': modified,
    })
  except:
    data.append({
      "Dataset ID": dataset_id,
      'Table': None,
      'Number of Rows': 0,
      'Number of Columns': 0,
      'Size in MB': None,
      'Last Modified': None,
    })


# Create a DataFrame from the list of data
table_df = pd.DataFrame(data)
table_df[:10]

In [ ]:
table_df.describe().transpose()


In [ ]:


# stop



## Create parquet from data frame


In [ ]:
parquet_file = "bigquery.dataset.metadata.parquet"
table_df.to_parquet( parquet_file, index=False)


In [ ]:
!ls -l big* --si


In [ ]:
duckdb.query(f"select count(1) from '{parquet_file}' ;")


In [ ]:
import duckdb
duckdb.query(f"select * from '{parquet_file}' limit 10;").df()


## Push parquet to AWS S3


In [ ]:
import os


In [ ]:
%%capture install_output
%%bash
pip install awscli


In [ ]:
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('aws_key')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('aws_secret')
os.environ['AWS_DEFAULT_REGION'] = 'us-west-1'


In [ ]:
!aws sts get-caller-identity

In [ ]:
!aws s3 cp {parquet_file} s3://ddc-datascience/{parquet_file}


In [ ]:
old_parquet_file = "bigquery.dataset.metadata.parquet"
parquet_file = "bigquery.dataset.metadata.tables.parquet"

!aws s3 mv s3://ddc-datascience/{old_parquet_file} s3://ddc-datascience/{parquet_file}


In [ ]:
url = f"https://ddc-datascience.s3.amazonaws.com/{parquet_file}"
duckdb.query(f"select 1 from '{url}' limit 1 ;")



In [ ]:
parquet_file = "bigquery.dataset.metadata.tables.parquet"
url = f"https://ddc-datascience.s3.amazonaws.com/{parquet_file}"
duckdb.query(f"select * from '{url}' limit 1 ;")


In [ ]:
import duckdb

parquet_file = "bigquery.dataset.metadata.tables.parquet"
url = f"https://ddc-datascience.s3.amazonaws.com/{parquet_file}"
duckdb.query(f"select * from '{url}' limit 1 ;")

In [ ]:
import duckdb

url_md_desc = f"https://ddc-datascience.s3.amazonaws.com/bigquery.dataset.metadata.description.parquet"
duckdb.query(f"select * from '{url_md_desc}' limit 1 ;").df()


In [ ]:

url_md_tables = f"https://ddc-datascience.s3.amazonaws.com/bigquery.dataset.metadata.tables.parquet"
duckdb.query(f"select * from '{url_md_tables}' limit 1 ;").df()


### Copy pickle to Google Drive

In [ ]:
from google.colab import drive
drive.mount('drive')


In [ ]:
!ls -la


In [ ]:
!ls -la drive/MyDrive/tmp/


In [ ]:
!cp bigquery.dataset.metadata.pkl drive/MyDrive/tmp/


## Load pickle archive from AWS

In [ ]:
url = "https://ddc-datascience.s3.amazonaws.com/bigquery.dataset.metadata.pkl"
df = pd.read_pickle( url )
df.shape

In [ ]:
df

### Number of tables in a data set

In [ ]:
df["Dataset"].value_counts().rename('Number of Tables')

### Size of data sets ( in GB ).

In [ ]:
(
  df
  .groupby(["Dataset"])["Size in MB"]
  .sum()/1000
).sort_values(ascending=False).rename('Size in GB')


### Size of tables ( in GB ) in each data set.

In [ ]:
( (df.groupby(["Dataset", "Table"])["Size in MB"].sum()/1000)
  .sort_values(ascending=False)
  .rename('Size in GB')
  .reset_index()
)